<a href="https://colab.research.google.com/github/malachi-nsona/aiclass/blob/main/Copy_of_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-openai -q
!pip install langchain-community -q
!pip install langchain-experimental -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 3.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from langchain_openai import OpenAIEmbeddings
class ZhipuAI_embeddings:
    def __init__(self, model_name: str = 'embedding-3'):
        self.model_name = model_name
        self.base_url = "https://open.bigmodel.cn/api/paas/v4"
        self.embedding = self._init_model()
    def _init_model(self) -> OpenAIEmbeddings:
        return OpenAIEmbeddings(
            model=self.model_name,
            base_url=self.base_url,
            api_key=userdata.get("apikey")
        )
embeddings =ZhipuAI_embeddings().embedding

In [3]:
from langchain_openai import ChatOpenAI
client  = ChatOpenAI(
    base_url ="https://open.bigmodel.cn/api/paas/v4/",
    api_key = userdata.get("apikey"),
    model = "glm-4.5"
)

In [4]:
client.invoke("hello")

AIMessage(content='Hello! How can I assist you today? 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 8, 'total_tokens': 21, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'glm-4.5', 'system_fingerprint': None, 'id': '2025091214201549d90358f8894b57', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--19a0e6d9-a5d9-4895-87ca-cda0752b48d2-0', usage_metadata={'input_tokens': 8, 'output_tokens': 13, 'total_tokens': 21, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [5]:
from langchain_core.tools import tool
@tool
def define_term(term:str):
  """give back definition of the term given by the user
  the terms could be Transformer or Embeddings,it just depends on the user
  term: a word or phrase to be defined
  {term}
  """

@tool
def summarize_notes(text:str):
  """you are to assist student in summarization of their notes
  convert each paragraph into concise summary of 2-3 sentences
  the summary should understandable,brief,clear and simple"""


In [6]:
client.bind_tools([define_term,summarize_notes])

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x7c0c7cd8bfb0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7c0c7ca1d520>, root_client=<openai.OpenAI object at 0x7c0c7d91b5f0>, root_async_client=<openai.AsyncOpenAI object at 0x7c0c7dafc500>, model_name='glm-4.5', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://open.bigmodel.cn/api/paas/v4/'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'define_term', 'description': 'give back definition of the term given by the user\n  the terms could be Transformer or Embeddings,it just depends on the user\n  term: a word or phrase to be defined\n  {term}', 'parameters': {'properties': {'term': {'type': 'string'}}, 'required': ['term'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'summarize_notes', 'description': 'you are to assist student in summarization of their notes\n  conv

In [7]:
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders import PyPDFLoader
def doc_parsing(file_path) -> list[Document]:
    if file_path.endswith("pdf"):
        doc = PyPDFLoader(file_path=file_path)
        doc = doc.load()
        semantic_splitter = SemanticChunker(
            embeddings=embeddings,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=95
        )
        full_text = doc[0].page_content if doc else ""
        if not full_text:
            return []
        raw_chunks = semantic_splitter.split_text(full_text)
        print(f" the number of chucks :{len(raw_chunks)}")
        docs = [Document(page_content=chunk, metadata=doc[0].metadata) for chunk in raw_chunks]
        return docs
    elif file_path.endswith(".txt"):
        doc = TextLoader(file_path=file_path)
        doc =doc.load()
        semantic_splitter = SemanticChunker(
            embeddings=embeddings,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=95
        )
        full_text = doc[0].page_content if doc else ""
        if not full_text:
            return []
        raw_chunks = semantic_splitter.split_text(full_text)
        docs = [Document(page_content=chunk, metadata=doc[0].metadata) for chunk in raw_chunks]
        return docs
    else:
       return []

In [8]:
s_template = """
Your name is AICLASS assistant a smart, yet sassy assistant working at takenolab, your have a better knowledge of takenolab operations,
your task is to be supportive, provide proper guidance to students that are having troubles with the course content,
you have to answer them direct and precise, if you dont have any advice to them dont generate any respose kindly
tell them so.
when given term to define ,call in define_term to give the correct definitions
summarise long notes for students when pasted using summarize_notes
When you receive:
• question: the student’s problem
• context: optionally, the content of any uploaded documents

If `context` is non-empty, you **must** use it to inform your answer. If you don’t have enough information
from the question and context combined, tell the student you can’t help further.

student problem:
{question}

uploaded document context (if any):
{context}

your smart advice or solution:

"""
new_template = """
Your name is AICLASS assistant, a smart, yet sassy assistant working at takenolab. You have a deep knowledge of takenolab operations.
Your task is to be supportive, provide proper guidance to students who are having trouble with the course content.
You must answer them directly and precisely. If you don't have any advice for them, kindly tell them so and do not generate any other response.

When you receive:
• chat_history: the previous turns of the conversation (if any)
• question: the student’s current problem
• context: optionally, the content of any uploaded documents relevant to the current question

If `context` is non-empty, you **must** use it to inform your answer. If you don’t have enough information
from the question and context combined, tell the student you can’t help further.
If `chat_history` is provided, use it to understand the full context of the current question.

<chat_history>
{chat_history}
</chat_history>

student problem:
{question}

uploaded document context (if any):
{context}

your smart advice or solution:

"""
SUB_QUERY_TEMPLATE = """
You are a helpful assistant that generates multiple search queries based on a single input query.
Generate {num_queries} diverse search queries related to the user's question, which can be used to retrieve relevant documents.
The queries should be concise and cover different aspects or angles of the original question.

Original Question: {question}

Generated Queries:
-

"""

In [9]:
from langchain_core.prompts import PromptTemplate
import gradio as gr
from langchain.chains import LLMChain
from langchain_core.documents import Document
from typing import TypedDict, List
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage
from typing import Optional,Tuple, Dict

vector_store = InMemoryVectorStore(embeddings)
def call_assistant(question:str="", context_docs:List[Optional[Document]]=None,chat_history: List[Tuple[str, str]]=None):
    prompt_template = PromptTemplate.from_template(template=new_template)
    chain = prompt_template | client
    history_str = ""
    if chat_history:
        for human_msg, ai_msg in chat_history:
            history_str += f"User: {human_msg}\n Assistant: {ai_msg}\n"
    context_text =""
    if context_docs:
         context_text = "\n\n".join(d.page_content for d in context_docs)
    response = chain.invoke({
        "question": question,
        "context": context_text,
        "chat_history": history_str
    })
    return response.content
def generate_sub_queries(original_question: str, num_queries: int = 3) -> List[str]:
    sub_query_prompt = PromptTemplate.from_template(template=SUB_QUERY_TEMPLATE)
    sub_query_chain = sub_query_prompt | client

    response = sub_query_chain.invoke({
        "question": original_question,
        "num_queries": num_queries
    })

    queries = [q.strip() for q in response.content.split('-') if q.strip()]
    print(f"Generated sub-queries: {queries}")
    return queries
def retrieve_and_answer_with_history(question: str, chat_history: List[Dict])->str:
    # retrieved = vector_store.similarity_search(question)
    formatted_chat_history_for_llm = []
    for msg in chat_history:
        if msg["role"] == "user":
            current_user_msg = msg["content"]
        elif msg["role"] == "assistant":
            formatted_chat_history_for_llm.append((current_user_msg, msg["content"]))
            current_user_msg = None
    if len(vector_store.store.items()) >= 0:
        transformed_queries = generate_sub_queries(question, num_queries=3)
        all_retrieved_docs = []
        seen_doc_contents = set()
        for query in transformed_queries:
            retrieved_for_query = vector_store.similarity_search(query)
            for doc in retrieved_for_query:
                if doc.page_content not in seen_doc_contents:
                    all_retrieved_docs.append(doc)
                    seen_doc_contents.add(doc.page_content)
        ai_response = call_assistant(question, context_docs=all_retrieved_docs, chat_history=formatted_chat_history_for_llm)
        return ai_response
    ai_response = call_assistant(question,chat_history=formatted_chat_history_for_llm)
    return ai_response
def doc_loader(file_path):
    docs = doc_parsing(file_path)
    if not docs:
        return "No content found or processed in the document."
    _ = vector_store.add_documents(documents=docs)
    return docs[0].page_content[:200] + "..."

def interface():
    iface = gr.ChatInterface(
        fn=retrieve_and_answer_with_history,
        chatbot=gr.Chatbot(height=200, type='messages', label="Assistant"),
        textbox=gr.Textbox(lines=2,submit_btn=True ),
        title="Takenolab AIClass Assistant (Conversational RAG)",
        type='messages',
        description="Ask a question about your course content and get smart advice, supporting multi-turn conversations.",
    )
    docs_interface = gr.Interface(
        fn=doc_loader,
        inputs=gr.File(label="Choose a file to upload",
                       type='filepath',
                       file_count='single',
                       show_label=True
                       ),
        description="Upload a document to run retrieval‐augmented generation.",
        outputs=gr.TextArea()
    )
    table = gr.TabbedInterface(
        [iface,docs_interface],
        tab_names= ['Chat', "Upload File for RAG"],
        title="LLM, RAG AND PROMPTS, Text Generation"
    )
    iface.launch(debug=True, server_port=3000)

In [10]:
client.invoke("what is an embedding").content

'An **embedding** is a way of representing discrete, categorical data (like words, products, users, or nodes in a network) as **dense, low-dimensional vectors** of real numbers. Think of it as a "translation" of complex, abstract entities into a language that machine learning models (especially neural networks) can understand and work with efficiently.\n\nHere\'s a breakdown of the key concepts:\n\n1.  **The Problem: Representing Categorical Data**\n    *   Traditional methods like **one-hot encoding** create sparse, high-dimensional vectors (e.g., a vector with 10,000 elements, all 0s except one 1 for a specific word in a vocabulary).\n    *   **Issues with one-hot:**\n        *   **High Dimensionality:** Vectors are huge, leading to computational inefficiency and the "curse of dimensionality."\n        *   **Sparsity:** Most elements are zero, making it hard for models to learn meaningful patterns.\n        *   **No Semantic Meaning:** The representation doesn\'t capture any inherent

In [ ]:
interface()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://139852b181a345ae53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generated sub-queries: ['Based on the minimal input "hi", here are 3 diverse search queries covering different aspects of greetings and communication:\n\n1.  **"common greetings in English"**  \n    *(Focuses on standard verbal greetings like "hi", "hello", "hey", exploring their usage and context.)*\n\n2.  **"professional email greeting alternatives"**  \n    *(Addresses the context of written communication, specifically looking for formal or varied ways to start emails beyond just "hi".)*\n\n3.  **"cultural differences in greeting customs"**  \n    *(Explores the broader anthropological and social aspect, examining how greetings (including simple ones like "hi") vary significantly across cultures and situations.)*']
Generated sub-queries: ['Here are 3 diverse search queries to help define "transformer," covering different key contexts:\n\n1.  **Electrical transformer definition components working principle**\n    *   *Focus:* Targets the classic electrical engineering context, seekin